## Import libraries and define necessary things

In [1]:
!pip install -Uq unsloth accelerate

In [2]:
import torch
import random
from PIL import Image
import matplotlib.pyplot as plt

In [3]:
SEED = 42

sys_prompt = """You're an assistant.
Think and answer carefully what's asked.
Give the final answer as \\boxed{answer} at last"""

random.seed(SEED)
torch.manual_seed(SEED)

## Load the Model and Processor

In [4]:
from unsloth import FastModel
from transformers import TextStreamer

model_name = "unsloth/gemma-4-E4B-it"

model, processor = FastModel.from_pretrained(
  model_name,
  load_in_4bit=True,
  use_gradient_checkpointing="unsloth",
  device_map = "cuda"
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

In [5]:
model = FastModel.get_peft_model(
     model,
     finetune_vision_layers=True,
     finetune_language_layers=True,
     finetune_attention_modules=True,
     finetune_mlp_modules=True,
     r=32,
     lora_alpha=32,
     lora_dropout=0,
     bias="none",
     random_state=SEED,
     use_rslora=False,
     loftq_config=None,
     target_modules="all-linear"
)

In [6]:
model.print_trainable_parameters()

trainable params: 82,444,288 || all params: 8,078,600,736 || trainable%: 1.0205


In [7]:
@torch.no_grad()
def chat(model, prompt, image=None, max_len=1024, think=False, temperature=1.0, top_p=0.95, top_k=64):
    user_content = []
    if image is not None:
          user_content.append({"type": "image", "image": image})

    user_content.append({"type": "text", "text": prompt.strip()})

    msg = [
        {"role": "system", "content": [{"type": "text", "text": sys_prompt}]},
        {"role": "user", "content": user_content}
    ]

    # Convert conversation to the model's specific string format
    inputs = processor.apply_chat_template(
        msg,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
        enable_thinking=think
    ).to("cuda")

    input_len = inputs["input_ids"].shape[1] # Corrected from shape[0] to shape[1] to get sequence length
    streamer = TextStreamer(processor.tokenizer, skip_prompt=True, skip_special_tokens=True)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_len,
        streamer=streamer,
        use_cache=True,
        do_sample=True,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p
    )
    #response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)
    # Parse output
    #return processor.parse_response(response)

In [8]:
chat(
  model,
  """If a student is asked to answer 20 MCQ each has 4 options.
  Find the probability of answering 12 questions correctly""",
  image=None,
  max_len=850,
  think=True
)

thought
Here's a thinking process to solve the problem:

1.  **Analyze the Request:** The user is asking for the probability of answering exactly 12 questions correctly out of 20 Multiple Choice Questions (MCQs), where each question has 4 options.

2.  **Identify the Type of Probability Distribution:**
    *   We have a fixed number of trials ($n=20$).
    *   Each trial (question) is independent of the others.
    *   Each trial has only two outcomes: Success (answering correctly) or Failure (answering incorrectly).
    *   The probability of success ($p$) is constant for every trial.
    *   This scenario fits the **Binomial Probability Distribution**.

3.  **Determine the Parameters:**
    *   Number of trials ($n$): 20
    *   Number of desired successes ($k$): 12
    *   Probability of success ($p$): Since there are 4 options and only 1 is correct, the probability of guessing correctly is $p = 1/4 = 0.25$.
    *   Probability of failure ($q$): $q = 1 - p = 1 - 0.25 = 0.75$.

4.  *

In [9]:
chat(
  model,
  prompt="""Complete the equation:
  H2S + O2 -> ?
  explain within 200 words!
  """,
  image=None,
  max_len=400,
  think=False,
  temperature=0.8,
  top_k=20
)

The reaction between hydrogen sulfide ($\text{H}_2\text{S}$) and oxygen ($\text{O}_2$) is a combustion reaction. In the presence of sufficient oxygen, hydrogen sulfide burns to produce sulfur dioxide ($\text{SO}_2$) and water ($\text{H}_2\text{O}$).

The unbalanced equation is:
$$\text{H}_2\text{S} + \text{O}_2 \rightarrow \text{SO}_2 + \text{H}_2\text{O}$$

To balance the equation, we need to ensure the number of atoms of each element is the same on both sides.

1. **Sulfur (S):** 1 on the left, 1 on the right (Balanced).
2. **Hydrogen (H):** 2 on the left, 2 on the right (Balanced).
3. **Oxygen (O):** 3 on the left ($2 \times 1.5$ if we consider $\text{H}_2\text{S}$ has 1 O, but we look at the coefficients), and $2 + 1 = 3$ on the right.

Let's check the oxygen atoms carefully:
*   Left side: 2 atoms in $\text{O}_2$.
*   Right side: 2 atoms in $\text{SO}_2$ + 1 atom in $\text{H}_2\text{O}$ = 3 atoms.

To balance the oxygen, we need to adjust the coefficients. If we use the coefficien

## Load the data

In [10]:
from datasets import load_dataset

ds, test_ds = load_dataset("derek-thomas/ScienceQA", split=["train", "test[:10]"])

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-1028f23e353fbe(…):   0%|          | 0.00/377M [00:00<?, ?B/s]

data/validation-00000-of-00001-6c7328ff6(…):   0%|          | 0.00/126M [00:00<?, ?B/s]

data/test-00000-of-00001-f0e719df791966f(…):   0%|          | 0.00/122M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12726 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4241 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4241 [00:00<?, ? examples/s]

In [11]:
ds

Dataset({
    features: ['image', 'question', 'choices', 'answer', 'hint', 'task', 'grade', 'subject', 'topic', 'category', 'skill', 'lecture', 'solution'],
    num_rows: 12726
})

In [12]:
def is_image_valid_and_convertible(example):
    # Ensure it's a PIL Image object
    if not isinstance(example["image"], Image.Image):
        return False
    try:
        # Attempt to convert to RGB and check basic properties
        # This will raise an error for truly corrupted images
        example["image"] = example["image"].convert("RGB")
        if example["image"].width == 0 or example["image"].height == 0:
            return False
        return True
    except Exception:
        # Catch any errors during conversion or property access
        return False

# Apply the more robust filter
ds = ds.filter(is_image_valid_and_convertible)

Filter:   0%|          | 0/12726 [00:00<?, ? examples/s]

In [13]:
def preprocess(exp):
  img = exp["image"].resize((256, 256))
  question = exp["question"]
  hint = exp["hint"]
  choices = exp["choices"]
  answer = choices[exp["answer"]]
  explanation = exp["solution"]
  idea = exp["lecture"]

  qsn = f"{hint if hint else ''}\n{question}".strip()

  # Randomly remove the idea part
  if random.random() < 0.5:
    idea = ''

  ans = f"Reasoning: {idea if idea else ''}\n{explanation if explanation else ''}\nAnswer: \\boxed{{{answer}}}".strip()

  user_content = [
   {"type": "image", "image": img},
   {"type": "text", "text": qsn}
  ]

  message = [
      {"role": "system", "content": sys_prompt},
      {"role": "user", "content": user_content},
      {"role": "assistant", "content": [
        {"type": "text", "text": ans}
        ]
      }
    ]
  return  message

In [14]:
ds = ds.shuffle(seed=SEED).take(3000)

In [15]:
processed_ds = [preprocess(exp) for exp in ds]

In [16]:
processed_ds[-1]

[{'role': 'system',
  'content': "You're an assistant.\nThink and answer carefully what's asked.\nGive the final answer as \\boxed{answer} at last"},
 {'role': 'user',
  'content': [{'type': 'image',
    'image': <PIL.Image.Image image mode=RGB size=256x256>},
   {'type': 'text',
    'text': 'The diagram below is a model of two solutions. Each blue ball represents one particle of solute.\nWhich solution has a higher concentration of blue particles?'}]},
 {'role': 'assistant',
  'content': [{'type': 'text',
    'text': 'Reasoning: \nIn Solution A and Solution B, the blue particles represent the solute. To figure out which solution has a higher concentration of blue particles, look at both the number of blue particles and the volume of the solvent in each container.\nUse the concentration formula to find the number of blue particles per milliliter.\nSolution B has more blue particles per milliliter. So, Solution B has a higher concentration of blue particles.\nAnswer: \\boxed{Solution B}

In [17]:
print(processor.apply_chat_template(processed_ds[-1], tokenize=False))

<bos><|turn>system
You're an assistant.
Think and answer carefully what's asked.
Give the final answer as \boxed{answer} at last<turn|>
<|turn>user


<|image|>

The diagram below is a model of two solutions. Each blue ball represents one particle of solute.
Which solution has a higher concentration of blue particles?<turn|>
<|turn>model
Reasoning: 
In Solution A and Solution B, the blue particles represent the solute. To figure out which solution has a higher concentration of blue particles, look at both the number of blue particles and the volume of the solvent in each container.
Use the concentration formula to find the number of blue particles per milliliter.
Solution B has more blue particles per milliliter. So, Solution B has a higher concentration of blue particles.
Answer: \boxed{Solution B}<turn|>



## Train the model(SFT) with QLoRA

In [18]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    train_dataset = processed_ds,
    processing_class = processor.tokenizer,
    data_collator = UnslothVisionDataCollator(model, processor),
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        dataloader_num_workers=4,
        max_grad_norm = 0.3,
        warmup_steps = 15,
        num_train_epochs = 2,
        learning_rate = 9e-5,
        logging_steps = 15,
        save_strategy = "steps",
        optim = "adamw_torch_fused",
        fp16=True,
        weight_decay = 0.001,
        lr_scheduler_type = "cosine",
        seed = SEED,
        output_dir = "outputs",
        report_to = "none", # For Weights and Biases or others
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,
        torch_empty_cache_steps=200
    )
)


trainer.train()

Unsloth: Model does not have a default image size - using 512


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,000 | Num Epochs = 2 | Total steps = 750
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 82,444,288 of 8,078,600,736 (1.02% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Step,Training Loss
15,9.245434
30,3.702342
45,2.504686
60,1.853941
75,1.611442
90,1.395873
105,1.176343
120,0.992751
135,0.903613
150,0.816470


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-750/tokenizer_config.json.


TrainOutput(global_step=750, training_loss=0.7717359107335409, metrics={'train_runtime': 11199.5578, 'train_samples_per_second': 0.536, 'train_steps_per_second': 0.067, 'total_flos': 9.019931978960486e+16, 'train_loss': 0.7717359107335409, 'epoch': 2.0})

## Test the model

In [19]:
def test_performance(data_idx):
  test = test_ds[data_idx]
  choice = test["choices"]

  chat(
    model,
    prompt=f'{test["question"]}\nOptions: {choice}',
    image=test["image"],
    think=True,
    temperature=1.0,
    top_p=0.95,
    top_k=64,
    max_len=1000
  )

  print("-"*30)
  print(f"Question: {test['question']}")
  print(f'Actual Ans: {choice[test["answer"]]}')

  if test["image"] is not None:
    plt.imshow(test["image"])
    plt.axis("off")
    plt.show()

In [20]:
test_performance(4)

thought
Thinking Process:

1.  **Analyze the Request:** The user wants me to find a word that would be located between two guide words in a dictionary. The guide words are "shot" and "suit".

2.  **Determine Alphabetical Order:** I need to place "shot" and "suit" in alphabetical order.
    *   s-h-o-t
    *   s-u-i-t
    *   "h" comes before "u". So, "shot" comes before "suit".

3.  **Establish the Range:** The target word must start with a letter that comes alphabetically after "shot" and before "suit".

4.  **Evaluate the Options:**
    *   'service': Starts with 's'. 's' comes after 'sh' in 'shot'. 's' comes before 'su' in 'suit'. So, 'service' fits between 'shot' and 'suit'.
    *   'stockade': Starts with 's'. 's' comes after 'sh' in 'shot'. 's' comes before 'su' in 'suit'. So, 'stockade' fits between 'shot' and 'suit'.

5.  **Re-evaluate the Goal:** The question asks for *a* word, and both options fit the criteria. However, in a multiple-choice scenario, usually only one answer i